# Validate a reference by donor

A reference of cellular states is only as good as its weakest donor. This tutorial builds a
reference, holds out one donor at a time, and compares the result with the conventional
random split over cells, which lets cells of the same donor sit on both sides of the split.

Everything runs on simulated data in seconds; the same calls apply to a real AnnData with a
donor column, a state column and a shared embedding in `obsm`.

In [1]:
import tempfile
import warnings
from pathlib import Path

import pandas as pd

import statetrace as st
from statetrace.datasets import simulate_transport

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)
workdir = Path(tempfile.mkdtemp(prefix="statetrace_tutorial_"))
# show warnings as one line, without the source file
warnings.formatwarning = (
    lambda message, category, *args, **kwargs: f"{category.__name__}: {message}\n"
)
st.__version__

'0.1.0'

## Simulated cohorts

`simulate_transport` returns a reference cohort, an independent query cohort (used in the
transport tutorial) and the planted truth per state. Here `S3` is a *minority* state: only a
quarter of the reference donors carry it.

In [2]:
ref_adata, query_adata, truth = simulate_transport(
    n_ref_groups=8,
    n_query_groups=6,
    n_states=5,
    n_latent=10,
    n_genes=300,
    shifted_states={"S1": 9.0},
    absent_states=["S4"],
    minority_states={"S3": 0.25},
    seed=5,
)
ref_adata

AnnData object with n_obs × n_vars = 1600 × 300
    obs: 'donor_id', 'cell_state', 'cell_type', 'batch', 'study'
    uns: 'statetrace_synthetic'
    obsm: 'X_latent'
    layers: None (.X)

In [3]:
truth

,truth,shift_magnitude,spread_factor,ref_donor_fraction
state,,,,
S0,stable,0.0,1.0,1.00
S1,shifted,9.0,1.0,1.00
S2,stable,0.0,1.0,1.00
S3,minority,0.0,1.0,0.25
S4,absent,0.0,1.0,1.00


In [4]:
pd.crosstab(ref_adata.obs["donor_id"], ref_adata.obs["cell_state"])

cell_state,S0,S1,S2,S3,S4
donor_id,,,,,
D00,41,40,33,38,48
D01,47,46,47,0,60
D02,52,45,57,0,46
D03,46,70,46,0,38
D04,41,54,45,0,60
D05,29,37,46,48,40
D06,44,41,58,0,57
D07,44,46,57,0,53


`S3` has many cells, but they come from two donors. A cell-level metric cannot see that.

## Build the reference

`group_key` is the replication unit (donors). `latent_key` points at the embedding in which
the built-in kNN baseline assigns states. With raw counts in `X`, StateTrace also builds a
frozen expression space from donor x state pseudobulk profiles, which does not depend on any
mapper's latent.

In [5]:
ref = st.Reference(
    ref_adata, group_key="donor_id", state_key="cell_state", latent_key="X_latent"
).fit()
ref

<StateReference 1600 cells, 8 groups, 5 states, 10-d latent, expression>

## Leave one donor out

One fold per donor: the backend is refitted on the other donors and scored on the held-out
one. Every state gets a reproducibility category (`stable`, `moderate`, `fragile` or
`insufficient`) from its per-donor recall.

In [6]:
res = ref.leave_one_group_out()
print(res.summary())

StateTrace leave_one_group_out over 'donor_id': 1600 cells, 8 groups, 5 states, 8 folds
  accuracy 0.999 | balanced accuracy 0.998 | macro F1 0.999 | abstention 0.000
  median group balanced accuracy 1.000
  states: stable=4, moderate=0, fragile=0, insufficient=1
  reproducibility judged in the expression space
  backend knn {'n_neighbors': 15, 'metric': 'cosine', 'weights': 'uniform', 'n_jobs': None}
  note: latent space was fixed across folds (fitted with every group present); this is an optimistic bound on embedding transfer.


In [7]:
res.state_metrics[
    [
        "n_groups_reference",
        "median_group_recall",
        "group_recovery_rate",
        "pooled_recall",
        "reproducibility",
    ]
]

,n_groups_reference,median_group_recall,group_recovery_rate,pooled_recall,reproducibility
state,,,,,
S0,8,1.000000,1.0,1.000000,stable
S1,8,1.000000,1.0,1.000000,stable
S2,8,1.000000,1.0,1.000000,stable
S3,2,0.986842,1.0,0.988372,insufficient
S4,8,1.000000,1.0,1.000000,stable


`S3` is recovered perfectly whenever it is seen, yet it is `insufficient`: two donors cannot
show that the state is reproducible across donors. The per-donor table behind these numbers
is `group_metrics`:

In [8]:
res.group_metrics.query("state == 'S3'")

,group,state,n_cells,n_predicted,recall,precision,f1,scored
3,D00,S3,38,37,0.973684,1.0,0.986667,True
24,D05,S3,48,48,1.000000,1.0,1.000000,True


## The conventional control

`random_split` runs a stratified K-fold over cells and ignores donors. It is the estimate most
papers report. Comparing it with the donor holdout makes the donor effect visible as a gap
instead of hiding it.

In [9]:
ctrl = ref.random_split(n_splits=5)
pd.DataFrame(
    {
        "cell_f1_random_split": ctrl.state_metrics["f1"],
        "f1_donor_holdout": res.state_metrics["f1"],
        "median_donor_recall": res.state_metrics["median_group_recall"],
        "reproducibility": res.state_metrics["reproducibility"],
    }
)

,cell_f1_random_split,f1_donor_holdout,median_donor_recall,reproducibility
state,,,,
S0,1.0,1.000000,1.000000,stable
S1,1.0,1.000000,1.000000,stable
S2,1.0,0.998716,1.000000,stable
S3,1.0,0.994152,0.986842,insufficient
S4,1.0,1.000000,1.000000,stable


On this easy simulation both estimates are near perfect, and the reproducibility category is
what separates `S3` from the other states. On real data the gap between the two columns is
often large; it is the optimism of cell-level evaluation.

## Keep the result

`to_dir` writes the result as plain JSON and CSV files that `ValidationResult.from_dir` reads
back. The validation also travels with the reference when it is frozen (next tutorial).

In [10]:
out = res.to_dir(workdir / "validation")
sorted(p.name for p in out.iterdir())

['cell_predictions.csv.gz',
 'config.json',
 'confusion_matrix.csv',
 'distance_calibration.csv',
 'group_metrics.csv',
 'group_summary.csv',
 'holdout_evidence.csv.gz',
 'overall.json',
 'state_metrics.csv']

```{note}
With the built-in kNN the latent space was fitted with every donor present, so the donor
holdout measures classifier transfer within a fixed embedding: an optimistic bound. The
summary says so, and every result records `latent["refit_latent"] = False`.
```